In [1]:
# ============================================================
# NOTEBOOK 3: MoE Evaluation × 3 Seeds — FINAL STEP
# ============================================================
import os
from pathlib import Path
import numpy as np
from PIL import Image
from ultralytics import YOLO

BASE      = "/home/modss/Desktop/dhanraj"
HIT_IOU   = 0.30
RCALC     = 0   # alphabetical: calc=0
RMASS     = 1   # mass=1
NC        = 4
NAMES     = ["benign_mass", "malignant_mass", "benign_calc", "malignant_calc"]
TEST_DIR  = Path(f"{BASE}/canonical_4class/test")
TEST_IMGS = sorted((TEST_DIR / "images").glob("*"))
TEST_LBL  = TEST_DIR / "labels"

# ----------------------------------------------------------
# HELPERS
# ----------------------------------------------------------

def load_gt(img_path):
    lbl = TEST_LBL / (Path(img_path).stem + ".txt")
    if not lbl.exists():
        return [], []
    with Image.open(img_path) as im:
        W, H = im.size
    boxes, labels = [], []
    for line in lbl.read_text().strip().splitlines():
        if not line.strip():
            continue
        c, cx, cy, bw, bh = map(float, line.split())
        x1 = (cx - bw/2) * W;  y1 = (cy - bh/2) * H
        x2 = (cx + bw/2) * W;  y2 = (cy + bh/2) * H
        boxes.append([x1, y1, x2, y2]);  labels.append(int(c))
    return boxes, labels

def box_iou_np(b1, b2):
    if not b1 or not b2:
        return np.zeros((len(b1), len(b2)))
    b1 = np.array(b1, dtype=np.float32)
    b2 = np.array(b2, dtype=np.float32)
    ix1 = np.maximum(b1[:,0:1], b2[:,0]);  iy1 = np.maximum(b1[:,1:2], b2[:,1])
    ix2 = np.minimum(b1[:,2:3], b2[:,2]);  iy2 = np.minimum(b1[:,3:4], b2[:,3])
    inter = np.maximum(ix2-ix1, 0) * np.maximum(iy2-iy1, 0)
    a1 = (b1[:,2]-b1[:,0]) * (b1[:,3]-b1[:,1])
    a2 = (b2[:,2]-b2[:,0]) * (b2[:,3]-b2[:,1])
    return inter / np.maximum(a1[:,None] + a2[None,:] - inter, 1e-7)

def compute_ap(recalls, precisions):
    r = np.concatenate([[0], recalls, [1]])
    p = np.concatenate([[0], precisions, [0]])
    for i in range(len(p)-2, -1, -1):
        p[i] = max(p[i], p[i+1])
    idx = np.where(r[1:] != r[:-1])[0]
    return float(np.sum((r[idx+1] - r[idx]) * p[idx+1]))

def _match_class(all_preds, all_gt_boxes, all_gt_cls, k, iou_thresh):
    flat = []
    for i, preds in enumerate(all_preds):
        for (c, x1, y1, x2, y2, cf) in preds:
            if c == k:
                flat.append((cf, i, x1, y1, x2, y2))
    flat.sort(key=lambda x: -x[0])
    total_gt = sum(1 for gcs in all_gt_cls for gc in gcs if gc == k)
    if total_gt == 0 or not flat:
        return np.array([]), np.array([]), total_gt
    matched = [set() for _ in range(len(all_gt_boxes))]
    tp, fp = [], []
    for (cf, i, x1, y1, x2, y2) in flat:
        gt_b = [b for b, gc in zip(all_gt_boxes[i], all_gt_cls[i]) if gc == k]
        gt_j = [j for j, gc in enumerate(all_gt_cls[i]) if gc == k]
        hit = False
        if gt_b:
            ious = box_iou_np([[x1,y1,x2,y2]], gt_b)[0]
            best = int(np.argmax(ious))
            if ious[best] >= iou_thresh and gt_j[best] not in matched[i]:
                matched[i].add(gt_j[best]);  hit = True
        tp.append(1 if hit else 0);  fp.append(0 if hit else 1)
    return np.array(tp), np.array(fp), total_gt

def compute_map_full(all_preds, all_gt_boxes, all_gt_cls):
    cls_ap50 = []
    for k in range(NC):
        tp, fp, tot = _match_class(all_preds, all_gt_boxes, all_gt_cls, k, 0.50)
        if tot == 0 or len(tp) == 0:
            cls_ap50.append(0.0); continue
        tpc = np.cumsum(tp);  fpc = np.cumsum(fp)
        cls_ap50.append(compute_ap(tpc/tot, tpc/(tpc+fpc)))
    map50 = float(np.mean(cls_ap50))
    aps = []
    for t in np.arange(0.50, 1.00, 0.05):
        per = []
        for k in range(NC):
            tp, fp, tot = _match_class(all_preds, all_gt_boxes, all_gt_cls, k, t)
            if tot == 0 or len(tp) == 0:
                per.append(0.0); continue
            tpc = np.cumsum(tp);  fpc = np.cumsum(fp)
            per.append(compute_ap(tpc/tot, tpc/(tpc+fpc)))
        aps.append(np.mean(per))
    return map50, float(np.mean(aps)), cls_ap50

def froc_eval(all_preds, all_gt_boxes, all_gt_cls, n_imgs):
    def _froc_k(k):
        tp, fp, tot = _match_class(all_preds, all_gt_boxes, all_gt_cls, k, HIT_IOU)
        if tot == 0 or len(tp) == 0:
            return 0.0
        tpc = np.cumsum(tp);  fpc = np.cumsum(fp)
        sens = tpc / tot;  fpi = fpc / n_imgs
        if fpi[-1] < 2.0:
            return float(sens[-1])
        if fpi[0] > 2.0:
            return 0.0
        return float(sens[np.searchsorted(fpi, 2.0)])
    per_cls = [_froc_k(k) for k in range(NC)]
    # Overall FROC on ALL predictions combined
    flat = [(cf, i, c, x1, y1, x2, y2)
            for i, preds in enumerate(all_preds)
            for (c, x1, y1, x2, y2, cf) in preds]
    flat.sort(key=lambda x: -x[0])
    total_gt = sum(len(g) for g in all_gt_cls)
    if total_gt == 0 or not flat:
        return 0.0, per_cls
    matched = [set() for _ in range(n_imgs)]
    tp, fp = [], []
    for (cf, i, c, x1, y1, x2, y2) in flat:
        gt_b = [b for b, gc in zip(all_gt_boxes[i], all_gt_cls[i]) if gc == c]
        gt_j = [j for j, gc in enumerate(all_gt_cls[i]) if gc == c]
        hit = False
        if gt_b:
            ious = box_iou_np([[x1,y1,x2,y2]], gt_b)[0]
            best = int(np.argmax(ious))
            if ious[best] >= HIT_IOU and gt_j[best] not in matched[i]:
                matched[i].add(gt_j[best]);  hit = True
        tp.append(1 if hit else 0);  fp.append(0 if hit else 1)
    tpc = np.cumsum(tp);  fpc = np.cumsum(fp)
    sens = tpc / total_gt;  fpi = fpc / n_imgs
    if fpi[-1] < 2.0:
        overall = float(sens[-1])
    elif fpi[0] > 2.0:
        overall = 0.0
    else:
        overall = float(sens[np.searchsorted(fpi, 2.0)])
    return overall, per_cls

def predict_detect(model, img_path, imgsz=1280):
    res = model.predict(str(img_path), imgsz=imgsz, conf=0.001, iou=0.6,
                        verbose=False, device=0)[0]
    preds = []
    if res.boxes is not None and len(res.boxes):
        for box in res.boxes:
            c  = int(box.cls.item())
            cf = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            preds.append((c, x1, y1, x2, y2, cf))
    return preds

def predict_router(model, img_path):
    res = model.predict(str(img_path), imgsz=640, verbose=False, device=0)[0]
    probs = res.probs.data.cpu().numpy()
    return float(probs[RCALC]), float(probs[RMASS])

# ----------------------------------------------------------
# MAIN EVAL LOOP
# ----------------------------------------------------------
n_imgs = len(TEST_IMGS)
print(f"Test images: {n_imgs}")

# Load ground truth once
all_gt_boxes, all_gt_cls = [], []
for img in TEST_IMGS:
    gb, gc = load_gt(img)
    all_gt_boxes.append(gb); all_gt_cls.append(gc)
print(f"Total GT boxes: {sum(len(g) for g in all_gt_cls)}")
for k in range(NC):
    print(f"  {NAMES[k]}: {sum(1 for gcs in all_gt_cls for c in gcs if c == k)}")

all_results = {}

for SEED in [42, 1, 123]:
    print(f"\n{'='*60}")
    print(f"SEED = {SEED}")
    print(f"{'='*60}")

    MASS_W   = f"{BASE}/canonical_mass_clean/runs/mass_clean_seed{SEED}/weights/best.pt"
    CALC_W   = f"{BASE}/canonical_calc_clean/runs/calc_clean_seed{SEED}/weights/best.pt"
    UNI_W    = f"{BASE}/canonical_4class/runs/universal_seed{SEED}/weights/best.pt"
    ROUTER_W = f"{BASE}/canonical_router_clean/runs/router_clean_seed{SEED}/weights/best.pt"

    mass_m   = YOLO(MASS_W)
    calc_m   = YOLO(CALC_W)
    uni_m    = YOLO(UNI_W)
    router_m = YOLO(ROUTER_W)

    print("Running inference on 458 test images...")
    mass_raw, calc_raw, uni_preds, rprobs = [], [], [], []
    for i, img in enumerate(TEST_IMGS):
        if i % 100 == 0:
            print(f"  {i}/{n_imgs}...")
        mass_raw.append(predict_detect(mass_m, img))
        calc_raw.append(predict_detect(calc_m, img))
        uni_preds.append(predict_detect(uni_m, img))
        rprobs.append(predict_router(router_m, img))

    # Remap calc expert: cls+2 (benign_calc→2, malignant_calc→3)
    mass_preds = mass_raw  # cls 0,1 → global 0,1 (no change)
    calc_preds = [[(c+2,x1,y1,x2,y2,cf) for (c,x1,y1,x2,y2,cf) in p] for p in calc_raw]

    # Domain-Split MoE: union
    ds_preds = [m + c for m, c in zip(mass_preds, calc_preds)]

    # Gated MoE: router-scaled confidence
    gated_preds = []
    for m, c, (p_calc, p_mass) in zip(mass_preds, calc_preds, rprobs):
        gm = [(cls,x1,y1,x2,y2,cf*p_mass) for (cls,x1,y1,x2,y2,cf) in m]
        gc = [(cls,x1,y1,x2,y2,cf*p_calc) for (cls,x1,y1,x2,y2,cf) in c]
        gated_preds.append(gm + gc)

    print("Computing mAP and FROC...")
    u_map50,  u_map5095,  u_cls_map  = compute_map_full(uni_preds,    all_gt_boxes, all_gt_cls)
    ds_map50, ds_map5095, ds_cls_map = compute_map_full(ds_preds,     all_gt_boxes, all_gt_cls)
    g_map50,  g_map5095,  g_cls_map  = compute_map_full(gated_preds,  all_gt_boxes, all_gt_cls)

    u_froc,  u_cls_froc  = froc_eval(uni_preds,   all_gt_boxes, all_gt_cls, n_imgs)
    ds_froc, ds_cls_froc = froc_eval(ds_preds,    all_gt_boxes, all_gt_cls, n_imgs)
    g_froc,  g_cls_froc  = froc_eval(gated_preds, all_gt_boxes, all_gt_cls, n_imgs)

    all_results[SEED] = {
        "universal":    {"map50": u_map50,  "map5095": u_map5095,  "froc": u_froc,  "cls_map": u_cls_map,  "cls_froc": u_cls_froc},
        "domain_split": {"map50": ds_map50, "map5095": ds_map5095, "froc": ds_froc, "cls_map": ds_cls_map, "cls_froc": ds_cls_froc},
        "gated":        {"map50": g_map50,  "map5095": g_map5095,  "froc": g_froc,  "cls_map": g_cls_map,  "cls_froc": g_cls_froc},
    }

    # Print clean result table
    print(f"\n{'─'*60}")
    print(f"SEED={SEED} | 458-img test | HIT_IOU={HIT_IOU} | mAP@IoU=0.50")
    print(f"{'─'*60}")
    print(f"{'Model':<20} {'mAP50':>7} {'mAP50-95':>9} {'FROC_S@2':>9}")
    print(f"{'─'*60}")
    for name, key in [("Universal",       "universal"),
                      ("Domain-Split MoE","domain_split"),
                      ("Gated MoE",       "gated")]:
        r = all_results[SEED][key]
        print(f"{name:<20} {r['map50']:>7.4f} {r['map5095']:>9.4f} {r['froc']:>9.4f}")

    print(f"\nPer-class mAP50 (seed={SEED}):")
    print(f"{'Model':<20} {'bm_mAP':>8} {'mm_mAP':>8} {'bc_mAP':>8} {'mc_mAP':>8}")
    print(f"{'─'*60}")
    for name, key in [("Universal","universal"),("Domain-Split","domain_split"),("Gated MoE","gated")]:
        cm = all_results[SEED][key]["cls_map"]
        print(f"{name:<20} {cm[0]:>8.4f} {cm[1]:>8.4f} {cm[2]:>8.4f} {cm[3]:>8.4f}")

    print(f"\nPer-class FROC S@2 (seed={SEED}):")
    print(f"{'Model':<20} {'bm_F':>8} {'mm_F':>8} {'bc_F':>8} {'mc_F':>8}")
    print(f"{'─'*60}")
    for name, key in [("Universal","universal"),("Domain-Split","domain_split"),("Gated MoE","gated")]:
        cf = all_results[SEED][key]["cls_froc"]
        print(f"{name:<20} {cf[0]:>8.4f} {cf[1]:>8.4f} {cf[2]:>8.4f} {cf[3]:>8.4f}")

# ----------------------------------------------------------
# MULTI-SEED SUMMARY
# ----------------------------------------------------------
print(f"\n{'='*60}")
print("MULTI-SEED SUMMARY (mean ± std over seeds 42, 1, 123)")
print(f"{'='*60}")
print(f"{'Model':<20} {'mAP50 mean±std':>18} {'FROC mean±std':>16}")
print(f"{'─'*60}")
for key, name in [("universal","Universal"),("domain_split","Domain-Split MoE"),("gated","Gated MoE")]:
    maps  = [all_results[s][key]["map50"] for s in [42, 1, 123]]
    frocs = [all_results[s][key]["froc"]  for s in [42, 1, 123]]
    print(f"{name:<20} {np.mean(maps):.4f}±{np.std(maps):.4f}       {np.mean(frocs):.4f}±{np.std(frocs):.4f}")

print(f"\nPer-class FROC S@2 mean (across 3 seeds):")
print(f"{'Model':<20} {'bm':>8} {'mm':>8} {'bc':>8} {'mc':>8}")
print(f"{'─'*60}")
for key, name in [("universal","Universal"),("domain_split","Domain-Split"),("gated","Gated MoE")]:
    per = np.mean([[all_results[s][key]["cls_froc"][k] for k in range(NC)] for s in [42,1,123]], axis=0)
    print(f"{name:<20} {per[0]:>8.4f} {per[1]:>8.4f} {per[2]:>8.4f} {per[3]:>8.4f}")

print("\n=== NOTEBOOK 3 COMPLETE — PASTE OUTPUT TO UPDATE RESULTS TABLE ===")

Test images: 458
Total GT boxes: 518
  benign_mass: 147
  malignant_mass: 106
  benign_calc: 164
  malignant_calc: 101

SEED = 42
Running inference on 458 test images...
  0/458...
  100/458...
  200/458...
  300/458...
  400/458...
Computing mAP and FROC...

────────────────────────────────────────────────────────────
SEED=42 | 458-img test | HIT_IOU=0.3 | mAP@IoU=0.50
────────────────────────────────────────────────────────────
Model                  mAP50  mAP50-95  FROC_S@2
────────────────────────────────────────────────────────────
Universal             0.0556    0.0239    0.2027
Domain-Split MoE      0.0519    0.0245    0.1873
Gated MoE             0.0484    0.0215    0.1737

Per-class mAP50 (seed=42):
Model                  bm_mAP   mm_mAP   bc_mAP   mc_mAP
────────────────────────────────────────────────────────────
Universal              0.0649   0.1185   0.0138   0.0253
Domain-Split           0.0672   0.0995   0.0129   0.0279
Gated MoE              0.0599   0.0833   0.0106  